In [0]:
%run ../00.common/1.environment_config

In [0]:
%run ../00.common/2.bronze_helper

In [0]:
target_table=f"{catalog_name}.{gold_schema}.fact_results"

In [0]:
from pyspark.sql import functions as F
results_df=(
    spark.table(f"{catalog_name}.{silver_schema}.results")
            .withColumn("session_type",F.lit("race"))
            .drop("race_name","race_date","ingestion_timestamp","source_file")

)

In [0]:
sprints_df=(
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
           .withColumn("session_type",F.lit("sprint"))
           .drop("race_name","race_date","ingestion_timestamp","source_file")
           .withColumnRenamed("completed_laps","laps_completed")
)

In [0]:
results_sprints_df=results_df.unionByName(sprints_df)

In [0]:
fact_session_results_df=(
    results_sprints_df
         .withColumn("is_win",F.col("finish_position")==1)
         .withColumn("is_podium",F.col("finish_position").between(1,3))
         .withColumn("has_points",F.col("points")>0)
)

In [0]:
(
    fact_session_results_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema","true")
        .saveAsTable(target_table)
)